# Projection and Residuals — Contextual Embeddings

Projects Hebrew and Arabic contextual embedding spaces onto the English space
using least-squares regression, then computes the residuals.

## What this does

For each foreign language (Hebrew, Arabic):
1. Find W such that `English @ W ≈ Foreign` (least-squares)
2. Compute projected: `English @ W`
3. Compute residual: `Foreign - projected`

The **residual** captures what Hebrew/Arabic encodes that English cannot explain.
This is the cross-lingual semantic signal tested in the encoding model.

## Input files
- `en_sliding_window_embeddings.csv` — (1735, 768) English contextual embeddings
- `he_sliding_window_embeddings.csv` — (1735, 768) Hebrew contextual embeddings
- `ar_sliding_window_embeddings.csv` — (1735, 768) Arabic contextual embeddings

## Output files
- `hebrew_residuals_contextual.npy`  — (1735, 768)
- `arabic_residuals_contextual.npy`  — (1735, 768)
- `hebrew_projected_contextual.npy`  — (1735, 768)
- `arabic_projected_contextual.npy`  — (1735, 768)

## 1. Load Embeddings

In [1]:
import numpy as np
import pandas as pd

DATA_DIR = '../data/processed/'

print('Loading contextual embeddings...')
E = pd.read_csv(DATA_DIR + 'en_sliding_window_embeddings.csv').values.astype(float)
H = pd.read_csv(DATA_DIR + 'he_sliding_window_embeddings.csv').values.astype(float)
A = pd.read_csv(DATA_DIR + 'ar_sliding_window_embeddings.csv').values.astype(float)

print(f'English : {E.shape}')
print(f'Hebrew  : {H.shape}')
print(f'Arabic  : {A.shape}')

assert E.shape == H.shape == A.shape, 'Shape mismatch between languages!'
assert E.shape[1] == 768, f'Expected 768 dims, got {E.shape[1]}'
print('\nAll shapes verified: (1735, 768) ✓')

Loading contextual embeddings...
English : (1735, 768)
Hebrew  : (1735, 768)
Arabic  : (1735, 768)

All shapes verified: (1735, 768) ✓


## 2. Projection and Residual

In [2]:
def project_and_residual(X_foreign, X_english):
    """
    Find W such that X_english @ W ≈ X_foreign (least squares).
    
    Projected = X_english @ W  (the part of foreign explainable by English)
    Residual  = X_foreign - Projected  (what English cannot explain)
    
    The residual is the cross-lingual semantic signal:
    information in Hebrew/Arabic that is orthogonal to English.
    """
    W, _, _, _ = np.linalg.lstsq(X_english, X_foreign, rcond=None)
    X_projected = X_english @ W
    X_residual  = X_foreign - X_projected
    return X_projected, X_residual


print('Computing Hebrew projection and residual...')
H_projected, H_residual = project_and_residual(H, E)
print(f'  Hebrew projected : {H_projected.shape}')
print(f'  Hebrew residual  : {H_residual.shape}')

print('\nComputing Arabic projection and residual...')
A_projected, A_residual = project_and_residual(A, E)
print(f'  Arabic projected : {A_projected.shape}')
print(f'  Arabic residual  : {A_residual.shape}')

Computing Hebrew projection and residual...
  Hebrew projected : (1735, 768)
  Hebrew residual  : (1735, 768)

Computing Arabic projection and residual...
  Arabic projected : (1735, 768)
  Arabic residual  : (1735, 768)


## 3. Sanity Checks

In [3]:
# The residual should be orthogonal to English (by construction)
# Check: correlation between English and residuals should be near 0

def mean_correlation(X, Y):
    """Mean absolute correlation between columns of X and Y."""
    X_norm = (X - X.mean(0)) / (X.std(0) + 1e-8)
    Y_norm = (Y - Y.mean(0)) / (Y.std(0) + 1e-8)
    corr = (X_norm * Y_norm).mean()
    return corr

print('Sanity checks:')
print(f'  Mean corr(English, Hebrew_residual)  : {mean_correlation(E, H_residual):.4f}  (should be ~0)')
print(f'  Mean corr(English, Arabic_residual)  : {mean_correlation(E, A_residual):.4f}  (should be ~0)')
print(f'  Mean corr(English, Hebrew)           : {mean_correlation(E, H):.4f}  (baseline)')
print(f'  Mean corr(English, Arabic)           : {mean_correlation(E, A):.4f}  (baseline)')

# Check variance explained by projection
he_var_explained = 1 - np.var(H_residual) / np.var(H)
ar_var_explained = 1 - np.var(A_residual) / np.var(A)
print(f'\n  Variance of Hebrew explained by English : {he_var_explained*100:.1f}%')
print(f'  Variance of Arabic explained by English : {ar_var_explained*100:.1f}%')
print(f'  (Residual contains the remaining {(1-he_var_explained)*100:.1f}% / {(1-ar_var_explained)*100:.1f}%)')

Sanity checks:
  Mean corr(English, Hebrew_residual)  : -0.0000  (should be ~0)
  Mean corr(English, Arabic_residual)  : -0.0000  (should be ~0)
  Mean corr(English, Hebrew)           : 0.3064  (baseline)
  Mean corr(English, Arabic)           : 0.2341  (baseline)

  Variance of Hebrew explained by English : 99.6%
  Variance of Arabic explained by English : 99.5%
  (Residual contains the remaining 0.4% / 0.5%)


## 4. Save

In [4]:
np.save(DATA_DIR + 'hebrew_residuals_contextual.npy',  H_residual)
np.save(DATA_DIR + 'arabic_residuals_contextual.npy',  A_residual)
np.save(DATA_DIR + 'hebrew_projected_contextual.npy',  H_projected)
np.save(DATA_DIR + 'arabic_projected_contextual.npy',  A_projected)

print('Saved:')
print(f'  hebrew_residuals_contextual.npy  {H_residual.shape}')
print(f'  arabic_residuals_contextual.npy  {A_residual.shape}')
print(f'  hebrew_projected_contextual.npy  {H_projected.shape}')
print(f'  arabic_projected_contextual.npy  {A_projected.shape}')
print('\nNext step: run 06_Encoding_Contextual.ipynb')

Saved:
  hebrew_residuals_contextual.npy  (1735, 768)
  arabic_residuals_contextual.npy  (1735, 768)
  hebrew_projected_contextual.npy  (1735, 768)
  arabic_projected_contextual.npy  (1735, 768)

Next step: run 06_Encoding_Contextual.ipynb
